# Model 2: Used car price prediction with multiple linear regression

### Authors
   * Jonathan Sancho Loaiza
   * Jennifer Lopez Miranda
   * Magdaleno Gomez Diaz

## Problem Analysis (CRISP-DM 1: Business Understanding)

A used car dealer in India buys cars from private owners and resells them, and the number that decides the margin of every deal is the price the car will fetch on the second-hand market. Today that number comes from the experience of the salesperson, who looks at the brand, the year, the mileage and the type of fuel and names a figure. The question this notebook answers is whether that figure can be produced by a model from the same facts, so that the assistant of the project can answer a question such as "how much is a 2015 diesel Fortuner with 40,000 km worth" with a number and a margin of error.

The dataset is the car data file published by Aman Kharwal, with 301 used cars sold in India. Each row records the selling price of the car, its showroom price when new, the year it was made, the kilometres it has driven, the fuel, the kind of seller, the transmission and how many previous owners it had. Prices are in lakhs of rupees, where one lakh is one hundred thousand rupees, so a selling price of 4.5 means 450,000 rupees.

The technique is multiple linear regression, the one taught in week four of the course. It fits a straight line through several predictors at once, so the selling price is written as a constant b0 plus a slope for each predictor, and the slopes tell in plain units how much each fact about the car is worth. That readability is the reason to prefer it over a black box here: the dealer wants to know that one more year of age costs so many lakhs, not only the final number.

### Libraries

In [ ]:
# Dataset handling
import os
import numpy as np
import pandas as pd

# Plots
import matplotlib.pyplot as plt
import seaborn as sns

# Model
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Split, evaluation and persistence
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

%matplotlib inline

# Fixed seed so the split and every number quoted in the text can be reproduced
RANDOM_STATE = 42

# Folder shared by every model of the project, read later by the REST API
MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

## Data Understanding (CRISP-DM 2)

The file is read straight from its public URL on GitHub, so the notebook runs without any manual download both on a local machine and in Google Colab.

In [ ]:
# Read the dataset straight from its public URL
DATA_URL = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/car%20data.csv'
car = pd.read_csv(DATA_URL)
car.head()

The first rows show one car per line. `Car_Name` is the model of the car, `Year` the year it was built, `Selling_Price` the price it sold for and the target of this notebook, `Present_Price` the price of the same car new in the showroom, `Kms_Driven` the mileage, and the last four columns describe the fuel, whether the seller was a dealer or an individual, the transmission and the number of previous owners. The first car, a Maruti Ritz from 2014, sold for 3.35 lakhs against a showroom price of 5.59.

In [ ]:
# Size of the table, column types and missing values
print("Rows and columns:", car.shape)
car.info()

There are 301 cars and 9 columns, and every column has 301 non null values, so there is nothing to impute. Four columns are text: `Car_Name`, `Fuel_Type`, `Seller_Type` and `Transmission`. A linear regression only multiplies numbers, so those four have to be turned into numbers or left out before the model is fitted, and that decision is taken in the preparation section.

In [ ]:
# Summary statistics of the numeric columns
car.describe().round(3)

The table gives, for each numeric column, the count, the mean, the standard deviation, the minimum, the three quartiles and the maximum. The quartiles split the sorted values in four equal parts, so the 50% row is the median, the value with half the cars below it.

The selling price has a mean of 4.66 lakhs but a median of 3.60, and the mean sitting well above the median means a few expensive cars pull it up: the maximum is 35 lakhs, ten times the median. The showroom price behaves the same way, with a median of 6.40 and a maximum of 92.60. The mileage has a median of 32,000 km and a maximum of 500,000 km, which is an outlier by any standard of a car that is on average five years old, since the years run from 2003 to 2018 with a median of 2014. `Owner` is almost always 0, with a mean of 0.043, so nearly every car is being sold by its first owner.

In [ ]:
# Duplicated rows and the categorical columns
print("Duplicated rows:", car.duplicated().sum())
print("Distinct car names:", car['Car_Name'].nunique())
for column in ['Fuel_Type', 'Seller_Type', 'Transmission', 'Owner']:
    print()
    print(car[column].value_counts())

Two rows are exact copies of another row. With 301 cars it is plausible that two identical cars were sold at the same price, but a copy that lands on both sides of the train and test split lets the model score a car it has already seen, so the copies are removed in the preparation section.

The categorical columns are unbalanced. Petrol has 239 cars, diesel 60 and CNG only 2, so whatever the model learns about CNG rests on two observations and should not be trusted. Dealers sold 195 cars and individuals 106. Manual cars are 261 against 40 automatic. `Owner` is 0 for 290 cars, 1 for ten and 3 for a single car. There are 98 distinct car names for 301 rows, an average of three cars per name, which is too few to learn a price per model, so `Car_Name` is dropped and the brand effect is left to `Present_Price`, which already encodes how expensive the car was new.

## Data Exploration (CRISP-DM 2) and Data Preparation (CRISP-DM 3)

The exploration answers two questions: how the selling price is distributed, because a linear regression works best when its errors are symmetric, and which of the other columns move together with it, because those are the predictors worth keeping.

In [ ]:
# Distribution of the target and of the two numeric predictors with a long tail
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, column in zip(axes, ['Selling_Price', 'Present_Price', 'Kms_Driven']):
    sns.histplot(car[column], bins=30, ax=ax, color='#4c72b0')
    ax.set_title(column)
plt.tight_layout()
plt.show()

# Imbalance of each column: 0 means both sides weigh the same, positive means
# the tail goes to the right, and beyond 1 the column is clearly unbalanced
print("Imbalance (skew) of Selling_Price:", round(car['Selling_Price'].skew(), 2))
print("Imbalance (skew) of Present_Price:", round(car['Present_Price'].skew(), 2))
print("Imbalance (skew) of Kms_Driven:", round(car['Kms_Driven'].skew(), 2))

Each histogram counts how many cars fall in each price or mileage band. The three of them pile up on the left and stretch a long tail to the right: most cars sold for less than 10 lakhs, a handful sold for 20 to 35, and the same happens with the showroom price and the kilometres.

The printed number is the skew, a measure of that imbalance. A value of 0 means the two sides of the histogram weigh the same, a positive value means the tail goes right, and past roughly 1 the column is clearly unbalanced. Here the selling price scores 2.49, the showroom price 4.08 and the mileage 6.44, so all three are strongly unbalanced. This matters for the technique: a regression fitted on such a target tends to make small errors on cheap cars and huge ones on expensive cars, and the residual analysis later checks whether that happened.

In [ ]:
# Average selling price by each categorical column
for column in ['Fuel_Type', 'Seller_Type', 'Transmission']:
    print(car.groupby(column)['Selling_Price'].agg(['mean', 'median', 'count']).round(2))
    print()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, column in zip(axes, ['Fuel_Type', 'Seller_Type', 'Transmission']):
    sns.boxplot(data=car, x=column, y='Selling_Price', ax=ax, color='#4c72b0')
    ax.set_title('Selling price by ' + column)
plt.tight_layout()
plt.show()

Each box plot shows the selling prices of one group. The box holds the middle half of the cars, the line inside is the median, the whiskers reach the values still close to the box and the dots beyond them are cars far from the rest of their group.

All three columns separate prices. Diesel cars have a median of 7.75 lakhs against 2.65 for petrol, dealers sell at a median of 5.25 against 0.52 for individuals, and automatics sit at 5.80 against 3.25 for manuals. The individual sellers are the sharpest split: their cars are mostly two-wheelers and cheap hatchbacks, which is why their median is under one lakh. These three columns carry information and are kept, each encoded as a 0 or 1 column.

In [ ]:
# Pearson measures how close the relationship is to a straight line, Spearman
# measures whether the ranking of one variable follows the ranking of the other
numeric = car[['Selling_Price', 'Present_Price', 'Year', 'Kms_Driven', 'Owner']]
print("Pearson correlation")
print(numeric.corr().round(3))
print()
print("Spearman correlation")
print(numeric.corr(method='spearman').round(3))

plt.figure(figsize=(7, 5))
sns.heatmap(numeric.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Pearson correlation between the numeric columns')
plt.show()

The heatmap colours each pair of columns by its correlation, red when both grow together and blue when one grows as the other falls, and the number in each cell is the coefficient. A correlation runs from -1 to 1, and 0 means no straight-line relationship at all.

The showroom price is by far the strongest predictor of the selling price, with a Pearson correlation of 0.88 and a Spearman of 0.91. Spearman being higher says the ranking is followed more faithfully than the straight line, which is the mark of a curved relationship where the expensive cars deviate from the line. `Year` correlates at 0.24 with the price, weaker than expected because the brand effect dominates: a three year old Fortuner costs more than a new Alto. The mileage is almost uncorrelated with the price at 0.03, and its Spearman of 0.15 is barely better, so kilometres will contribute little once the other columns are in. `Owner` is negative at -0.09, consistent with the second-hand value dropping with each owner, but it rests on eleven cars.

No pair of predictors exceeds 0.75, the alarm threshold for multicollinearity used in the course. The strongest pair among predictors is `Year` against `Kms_Driven` at -0.52, which is the obvious fact that older cars have driven more, and it is not high enough to break the model. The VIF computed after encoding confirms this.

In [ ]:
# Scatter of the two strongest numeric predictors against the target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=car, x='Present_Price', y='Selling_Price', hue='Fuel_Type', ax=axes[0])
axes[0].set_title('Selling price against showroom price')
sns.scatterplot(data=car, x='Year', y='Selling_Price', hue='Seller_Type', ax=axes[1])
axes[1].set_title('Selling price against year')
plt.tight_layout()
plt.show()

The left panel plots every car by its showroom price and its selling price. The points follow a line up to about 20 lakhs of showroom price and then fan out, and the one point far to the right is the Land Cruiser, a 92.6 lakh car that sold for 35. That single car sits alone in a region of the plot where the line has nothing else to lean on, and the residual analysis will show whether the model handles it.

The right panel shows the year. Prices rise with the year, but inside each year the spread is huge because the cheap cars of individual sellers, drawn in orange, sit at the bottom of every year while the dealer cars spread upward. That is why `Year` alone correlates weakly and why it needs `Present_Price` beside it to become useful.

In [ ]:
# Remove the duplicated rows
car = car.drop_duplicates().reset_index(drop=True)
print("Rows after removing duplicates:", len(car))

# Age of the car in years, counted from the last year present in the dataset,
# so the slope reads as lakhs lost per year instead of lakhs gained per year
REFERENCE_YEAR = car['Year'].max()
car['Age'] = REFERENCE_YEAR - car['Year']

# Categorical columns turned into 0 or 1 columns. Petrol, Dealer and Manual are
# the reference groups, so each slope reads as the difference against them
car['Fuel_Diesel'] = (car['Fuel_Type'] == 'Diesel').astype(int)
car['Fuel_CNG'] = (car['Fuel_Type'] == 'CNG').astype(int)
car['Seller_Individual'] = (car['Seller_Type'] == 'Individual').astype(int)
car['Transmission_Automatic'] = (car['Transmission'] == 'Automatic').astype(int)

FEATURES = ['Present_Price', 'Age', 'Kms_Driven', 'Owner',
            'Fuel_Diesel', 'Fuel_CNG', 'Seller_Individual', 'Transmission_Automatic']
TARGET = 'Selling_Price'
car[FEATURES + [TARGET]].head()

The table now has the eight predictors the model will read. `Age` replaces `Year` so that its slope is a depreciation per year, which is how a dealer thinks. The four encoded columns are 1 when the car belongs to the group in the column name and 0 otherwise, and the group left out of each pair, petrol, dealer and manual, is the reference the slope is compared against.

In [ ]:
# Variance inflation factor of each predictor. It measures how well a predictor
# can be predicted from all the others: 1 means not at all, 10 is the alarm level
X_all = sm.add_constant(car[FEATURES])
vif_table = pd.DataFrame({
    'Predictor': FEATURES,
    'VIF': [variance_inflation_factor(X_all.values, i + 1) for i in range(len(FEATURES))]
}).round(2)
print(vif_table)

The VIF of a predictor is how much its own variance is inflated by its correlation with the rest, and a value of 10 is the point where the slopes stop being trustworthy. Every predictor here sits below 2, with `Present_Price` the highest at 1.81. The correlation matrix saw no pair above 0.75 and the VIF sees no predictor explained by the group, so multicollinearity is not a problem for this model and every slope can be read on its own.

In [ ]:
# 75 / 25 split with a fixed seed
car_train, car_test = train_test_split(car, test_size=0.25, random_state=RANDOM_STATE)
print("Training cars:", len(car_train))
print("Test cars:", len(car_test))

## Machine Learning Model (CRISP-DM 4: Modeling)

Two models are fitted. The first uses the selling price and the predictors as they come, which is the literal reading of the task. The second predicts the logarithm of the price, because the exploration found the price and the showroom price strongly unbalanced and the residuals of the first model, shown below, confirm that the straight line struggles with the expensive cars. Both are evaluated on the same test cars with the same metrics, and the results section states which one is kept and what the choice costs.

### Model A: linear regression on the raw price

In [ ]:
# Ordinary least squares on the raw target, with a constant so b0 is fitted
X_train = sm.add_constant(car_train[FEATURES])
y_train = car_train[TARGET]
model_raw = sm.OLS(y_train, X_train).fit()
print(model_raw.summary())

The summary table has three blocks. The top block reports how much of the variation of the price the model explains: R-squared is the share of the variance explained, from 0 to 1, and the adjusted R-squared lowers it a little for every predictor added, so that a useless predictor does not raise the score. The middle block has one row per predictor with its slope in the `coef` column, its uncertainty in `std err`, and `P>|t|`, which is the probability of seeing a slope that large if the true slope were zero. The bottom block describes the residuals, the differences between the real and the predicted prices.

On the training cars the model reaches an R-squared of 0.906 and an adjusted one of 0.903, so it explains 90 percent of the variation. The slopes read in lakhs. Each lakh of showroom price adds 0.44 lakhs to the selling price, each year of age removes 0.32, a diesel engine adds 1.46 and an automatic gearbox 1.62, and a car sold by an individual sells for 1.33 lakhs less than the same car sold by a dealer. Every one of those has the sign the exploration predicted. The mileage slope is tiny, about one lakh per 87,000 km, though its `P>|t|` of 0.030 says it is distinguishable from zero. The owner slope is positive, which contradicts its negative correlation with the price, and its `P>|t|` of 0.319 says it is noise from eleven cars. The CNG slope rests on two cars and has a `P>|t|` of 0.655, so it adds nothing either.

The bottom block is the warning. The skew of the residuals is 0.80 and the kurtosis 7.19, where a symmetric bell would give 0 and 3, so the errors have a long tail of large misses. The next cells look at where those misses come from.

In [ ]:
# Standardized residuals: each residual divided by its own standard deviation,
# so a value beyond 3 marks a car the model missed badly
residuals_raw = model_raw.get_influence().resid_studentized_internal

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(residuals_raw, bins=30, ax=axes[0], color='#4c72b0')
axes[0].set_title('Histogram of standardized residuals')
sns.boxplot(x=residuals_raw, ax=axes[1], color='#4c72b0')
axes[1].set_title('Box plot of standardized residuals')
sm.qqplot(residuals_raw, line='s', ax=axes[2])
axes[2].set_title('Q-Q plot of standardized residuals')
plt.tight_layout()
plt.show()

# Residual against the fitted value shows whether the error grows with the price
plt.figure(figsize=(7, 4))
plt.scatter(model_raw.fittedvalues, residuals_raw, alpha=0.6)
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Predicted selling price (lakhs)')
plt.ylabel('Standardized residual')
plt.title('Residuals against predicted price, raw model')
plt.show()
print("Training cars with a standardized residual beyond 3:", int((abs(residuals_raw) > 3).sum()))

The three panels are the standard check of the residuals. The histogram should look like a bell centred on 0. The box plot should be a short box with few dots beyond the whiskers. The Q-Q plot compares the sorted residuals against what a normal distribution would produce, and if the residuals were normal the dots would sit on the red line.

None of the three looks the way it should. The histogram is tall in the middle with a long tail on both sides, the box plot shows several dots far beyond the whiskers, and in the Q-Q plot the dots leave the line at both ends, which is the signature of heavy tails. The fourth plot explains why: for cheap cars, on the left, the residuals hug zero, but as the predicted price grows the residuals spread out in a fan. The spread of the errors grows with the prediction, so the assumption of a constant spread, which the course calls homocedasticidad, is violated. In practice this means the model makes a one lakh error on a cheap car and a five lakh error on an expensive one, and a fixed margin of error cannot be given.

The course explains that a target with a long right tail is usually fixed by taking its logarithm, which compresses the large values and turns a multiplicative relationship, where an expensive car loses a percentage of its value per year, into an additive one, which is what a straight line can fit. That is Model B.

### Model B: linear regression on the logarithm of the price

In [ ]:
# Logarithm of the target and of the two unbalanced predictors. log1p is used for
# the mileage because it handles a value of 0 km
def to_log_space(frame):
    transformed = frame[FEATURES].copy()
    transformed['Present_Price'] = np.log(frame['Present_Price'])
    transformed['Kms_Driven'] = np.log1p(frame['Kms_Driven'])
    return transformed

X_train_log = sm.add_constant(to_log_space(car_train))
y_train_log = np.log(car_train[TARGET])
model_log = sm.OLS(y_train_log, X_train_log).fit()
print(model_log.summary())

The same table for the log model. The R-squared rises from 0.906 to 0.982 on the training cars, and the residual block improves with it: the skew drops from 0.80 to -0.04 and the kurtosis from 7.19 to 3.87, close to the 3 of a normal bell. The slopes are now in log units, which read as percentages. A slope of s on a 0 or 1 column means the group sells for roughly (exp(s) - 1) times 100 percent more, so the diesel slope of 0.12 says a diesel sells for about 13 percent more than a petrol car of the same showroom price and age, the age slope of -0.095 says each year removes about 9 percent of the price, and the owner slope of -0.22 now has the negative sign its correlation asked for. The `Present_Price` slope of 0.95, close to 1 in log space, means the selling price scales almost proportionally with the showroom price, which is what a percentage of depreciation implies.

Two slopes changed status. The mileage `P>|t|` is now 0.000 with a slope of -0.076 per unit of log kilometres, so once the price is in log space the mileage carries a clear effect. The automatic transmission slope, worth 1.62 lakhs in the raw model, is now -0.0015 with a `P>|t|` of 0.968: automatic cars are expensive cars, and once the price is proportional to the showroom price the gearbox adds nothing by itself. The raw model was reading the price level as a gearbox effect.

In [ ]:
# Same three-panel check on the log model
residuals_log = model_log.get_influence().resid_studentized_internal

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(residuals_log, bins=30, ax=axes[0], color='#4c72b0')
axes[0].set_title('Histogram of standardized residuals')
sns.boxplot(x=residuals_log, ax=axes[1], color='#4c72b0')
axes[1].set_title('Box plot of standardized residuals')
sm.qqplot(residuals_log, line='s', ax=axes[2])
axes[2].set_title('Q-Q plot of standardized residuals')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(model_log.fittedvalues, residuals_log, alpha=0.6)
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Predicted log of selling price')
plt.ylabel('Standardized residual')
plt.title('Residuals against predicted value, log model')
plt.show()
print("Training cars with a standardized residual beyond 3:", int((abs(residuals_log) > 3).sum()))

The histogram is now a bell centred on zero, the box plot is compact with only a few dots outside the whiskers, and the Q-Q dots follow the line except for a couple at each end. The residual against prediction plot no longer fans out: the spread is about the same for cheap and expensive cars, so the constant spread assumption now holds and a percentage margin of error can be given for any car. The count of training cars beyond 3 standard deviations drops from 5 to 2, and those two are cars whose price does not follow their showroom price. They are worth listing at the end rather than removing, since nothing says they are errors.

## Evaluation (CRISP-DM 5)

Both models predict the same test cars, the 25 percent held back before fitting. Test cars whose predictors fall outside the range the model was trained on are removed first, because a straight line has no information beyond the range of its training data and predicting there is extrapolation. The metrics are R-squared, the mean absolute error, which is the average miss in lakhs, and the root mean squared error, which weighs large misses more heavily.

In [ ]:
# Remove test cars outside the training range of any numeric predictor
NUMERIC_FEATURES = ['Present_Price', 'Age', 'Kms_Driven', 'Owner']
inside = pd.Series(True, index=car_test.index)
for column in NUMERIC_FEATURES:
    low, high = car_train[column].min(), car_train[column].max()
    inside &= car_test[column].between(low, high)
removed = int((~inside).sum())
car_test_inside = car_test[inside]
print("Test cars removed for falling outside the training range:", removed)
print("Test cars kept:", len(car_test_inside))

In [ ]:
# Predictions of both models on the same test cars
y_test = car_test_inside[TARGET]
pred_raw = model_raw.predict(sm.add_constant(car_test_inside[FEATURES]))
pred_log = np.exp(model_log.predict(sm.add_constant(to_log_space(car_test_inside))))

def report(name, y_true, y_pred, n_predictors):
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    adjusted = 1 - (1 - r2) * (n - 1) / (n - n_predictors - 1)
    return pd.Series({
        'R2': r2, 'Adjusted R2': adjusted,
        'MAE (lakhs)': mean_absolute_error(y_true, y_pred),
        'RMSE (lakhs)': np.sqrt(mean_squared_error(y_true, y_pred)),
    }, name=name)

comparison = pd.concat([report('Model A: raw price', y_test, pred_raw, len(FEATURES)),
                        report('Model B: log price', y_test, pred_log, len(FEATURES))], axis=1).round(3)
print(comparison)

The comparison is measured in the units of the problem, lakhs of rupees, on the same cars. Three test cars fell outside the training range and were removed, leaving 72. Model B beats Model A on every metric: its R-squared on unseen cars is 0.954 against 0.807, its average miss is 0.65 lakhs against 1.23, and its root mean squared error, which punishes the large misses, is 1.03 against 2.11, which is exactly the effect the residual plots predicted. Model A is not a useless model, an R-squared of 0.81 on unseen cars still explains most of the price, but for a dealer the average error is the number that matters and Model B roughly halves it, from 123,000 rupees to 65,000.

In [ ]:
# Real against predicted price for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, pred) in zip(axes, [('Model A: raw price', pred_raw), ('Model B: log price', pred_log)]):
    ax.scatter(y_test, pred, alpha=0.6)
    limit = max(y_test.max(), pred.max())
    ax.plot([0, limit], [0, limit], color='red', linewidth=1)
    ax.set_xlabel('Real selling price (lakhs)')
    ax.set_ylabel('Predicted selling price (lakhs)')
    ax.set_title(name)
plt.tight_layout()
plt.show()

Each dot is a test car placed by its real price and its predicted price, and the red diagonal is where a perfect prediction would fall. In the left panel the dots of the cheap cars sit on the line but the expensive ones scatter above and below it, and some cheap cars are predicted below zero, which a raw linear model can do because nothing stops the line from going negative. In the right panel the dots hug the diagonal along the whole range and no prediction is negative, since the exponential of any number is positive.

In [ ]:
# The test cars that Model B misses the most
errors = car_test_inside[['Car_Name', 'Year', 'Fuel_Type', 'Seller_Type', 'Present_Price', TARGET]].copy()
errors['Predicted'] = pred_log.round(2)
errors['Error'] = (errors['Predicted'] - errors[TARGET]).round(2)
errors.reindex(errors['Error'].abs().sort_values(ascending=False).index).head(5)

Four of the five largest misses are the same car, a Toyota Corolla Altis with a showroom price near 20 lakhs that sold for between 4 and 7, and the model predicts 7 to 10 for it. A large petrol sedan loses value faster than its showroom price alone predicts, and with only a handful of them in the data the model cannot learn that. The fifth is an Innova predicted 2.7 lakhs below its real price. These rows are kept in the evaluation. They are real sales and removing them would only make the metrics look better than the dealer will experience.

## Results (CRISP-DM 6: Deployment)

The model kept is Model B, a multiple linear regression fitted on the logarithm of the selling price with the logarithm of the showroom price, the logarithm of the mileage, the age, the number of owners and the encoded fuel, seller and transmission as predictors. On the 72 test cars inside the training range it reaches an R-squared of 0.954, a mean absolute error of 0.65 lakhs and a root mean squared error of 1.03, against 0.807, 1.23 and 2.11 for the raw model. The task asked for a linear regression on the price and that model was built and evaluated first. The deviation to log space is defended by the residual plots: the raw model's errors fan out with the price and its residuals have heavy tails, both of which disappear with the transformation, and the cost of the change is that the slopes now read as percentages instead of lakhs, which for a depreciation problem is the more natural reading anyway.

The model is recommended for the assistant with two limitations stated plainly. First, it was trained on cars up to the showroom price and age present in the training set, and any car outside that range is an extrapolation the line knows nothing about, which the evaluation enforced by removing such cars. Second, the CNG slope rests on two cars and should be read as unknown rather than as a number. The model is saved below together with the list of predictors and the transformation, so the REST API can load it and predict without training again.

In [ ]:
# Persist the chosen model with everything the API needs to build its input
bundle = {
    'model': model_log,
    'features': FEATURES,
    'target': TARGET,
    'reference_year': int(REFERENCE_YEAR),
    'log_features': ['Present_Price'],
    'log1p_features': ['Kms_Driven'],
    'target_transform': 'log',
    'training_range': {c: (float(car_train[c].min()), float(car_train[c].max())) for c in NUMERIC_FEATURES},
}
MODEL_PATH = os.path.join(MODELS_DIR, 'car_price_regression.joblib')
joblib.dump(bundle, MODEL_PATH)
print("Saved to", MODEL_PATH)

In [ ]:
# Reload the model and predict one car by hand: a 2015 diesel car sold by a
# dealer, manual gearbox, 40,000 km, showroom price 10 lakhs, first owner
loaded = joblib.load(MODEL_PATH)
example = pd.DataFrame([{
    'Present_Price': 10.0, 'Age': loaded['reference_year'] - 2015, 'Kms_Driven': 40000, 'Owner': 0,
    'Fuel_Diesel': 1, 'Fuel_CNG': 0, 'Seller_Individual': 0, 'Transmission_Automatic': 0,
}])
example_log = example.copy()
example_log['Present_Price'] = np.log(example_log['Present_Price'])
example_log['Kms_Driven'] = np.log1p(example_log['Kms_Driven'])
predicted = np.exp(loaded['model'].predict(sm.add_constant(example_log, has_constant='add')))[0]
print("Predicted selling price: %.2f lakhs" % predicted)

The reloaded model prices a three year old diesel car with a showroom price of 10 lakhs at 7.34 lakhs, which sits where the exploration placed diesel dealer cars of that age, a median of 7.75. This is the call the REST API will make: build the same eight columns, apply the two logarithms, predict and take the exponential.